# Rescuing Leftover Cuisine — Data analysis (JPMC Hackathon)

This notebook turns the rescue log into **trends**, **partner insights**, **zip-level opportunities**, and **scheduling-oriented signals** aligned with the challenge brief (NYC, Boston, Chicago, LA).

**Dataset:** `all_rescures_all_time_final_nyc_boston_la_chi.csv` (~155k rows). **Dictionary:** see project PDF (18 fields).

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 5)

DATA_PATH = Path("all_rescures_all_time_final_nyc_boston_la_chi.csv")
OUT_DIR = Path("figures")
OUT_DIR.mkdir(exist_ok=True)

## 1. Load and clean
- Parse dates; coerce `Pounds Rescued` to numeric.
- **Normalize donor city** (fixes `boston` / ` Chicago` / etc.).
- For impact metrics, focus on **`Rescue is Finished == True`** (per data dictionary).
- Optional: exclude future-dated rows from *historical* trend charts (dataset contains scheduled-looking future dates).

In [2]:
def normalize_donor_city(s):
    if pd.isna(s):
        return pd.NA
    x = str(s).strip().lower()
    x = " ".join(x.split())
    if "new york" in x or x in ("nyc",):
        return "New York City"
    if "boston" in x:
        return "Boston"
    if "chicago" in x:
        return "Chicago"
    if "los angeles" in x or x == "la":
        return "Los Angeles"
    return str(s).strip().title()


raw = pd.read_csv(DATA_PATH, low_memory=False)
raw["Rescue Date"] = pd.to_datetime(raw["Rescue Date"], errors="coerce")
raw["Pounds Rescued"] = pd.to_numeric(raw["Pounds Rescued"], errors="coerce")
raw["Donor City"] = raw["Donor City"].map(normalize_donor_city)

done = raw[raw["Rescue is Finished"]].copy()
done = done[done["Pounds Rescued"].notna() & (done["Pounds Rescued"] >= 0)]

as_of = pd.Timestamp.today().normalize()
hist = done[done["Rescue Date"] <= as_of].copy()

print(f"Rows (all): {len(raw):,}")
print(f"Finished rescues: {len(done):,}")
print(f"Finished through today ({as_of.date()}): {len(hist):,}")
print("Donor cities:", raw["Donor City"].value_counts().to_dict())

Rows (all): 155,225
Finished rescues: 96,985
Finished through today (2026-04-10): 96,937
Donor cities: {'Boston': 106067, 'Chicago': 35864, 'Los Angeles': 6855, 'New York City': 6439}


## 2. Executive KPIs (finished rescues, historical)

In [3]:
kpi = hist.groupby("Donor City", dropna=False).agg(
    rescues=("Pounds Rescued", "count"),
    total_lbs=("Pounds Rescued", "sum"),
    median_lbs=("Pounds Rescued", "median"),
).sort_values("total_lbs", ascending=False)
kpi["lbs_per_rescue"] = kpi["total_lbs"] / kpi["rescues"]
display(kpi.round(1))

completion_rate = raw["Rescue is Finished"].mean()
print(f"Overall completion rate (all rows): {completion_rate:.1%}")

,rescues,total_lbs,median_lbs,lbs_per_rescue
Donor City,,,,
Boston,63846,3273514.0,50.0,51.3
Chicago,23540,3072624.0,44.0,130.5
Los Angeles,4564,173863.0,30.0,38.1
New York City,4987,159861.0,30.0,32.1


Overall completion rate (all rows): 62.5%


## 3. Where & when: volume over time (by city)
Monthly **pounds** and **rescue counts** show growth, seasonality, and city mix.

In [4]:
hist["year_month"] = hist["Rescue Date"].dt.to_period("M").dt.to_timestamp()
month_city = (
    hist.groupby(["year_month", "Donor City"], dropna=False)["Pounds Rescued"]
    .sum()
    .reset_index()
)

fig, ax = plt.subplots()
for city in ["Boston", "Chicago", "Los Angeles", "New York City"]:
    sub = month_city[month_city["Donor City"] == city]
    ax.plot(sub["year_month"], sub["Pounds Rescued"] / 1e3, label=city, linewidth=2)
ax.set_title("Finished rescues: food volume by city (monthly, thousands of lbs)")
ax.set_ylabel("Thousand lbs")
ax.legend(loc="upper left", fontsize=10)
plt.tight_layout()
fig.savefig(OUT_DIR / "01_monthly_pounds_by_city.png", dpi=150)
plt.show()

/var/folders/jb/prnhrlmn6ln8gp40kvxptyq80000gn/T/ipykernel_71453/182265625.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Seasonality (calendar month)
Average **index** = 100 at the overall mean; highlights months that systematically run above/below typical volume (helps staffing and donor outreach timing).

In [5]:
hist["cal_month"] = hist["Rescue Date"].dt.month
city_month = hist.groupby(["Donor City", "cal_month"])["Pounds Rescued"].sum().reset_index()
overall_month = hist.groupby("cal_month")["Pounds Rescued"].sum()
overall_mean = overall_month.mean()
season_idx = (overall_month / overall_mean * 100).reindex(range(1, 13))

fig, ax = plt.subplots()
ax.bar(season_idx.index, season_idx.values, color="steelblue", alpha=0.85)
ax.axhline(100, color="black", linestyle="--", linewidth=1)
ax.set_xticks(range(1, 13))
ax.set_xlabel("Month")
ax.set_ylabel("Index (100 = average month)")
ax.set_title("Seasonality — total lbs by calendar month (all cities, finished)")
plt.tight_layout()
fig.savefig(OUT_DIR / "02_seasonality_index.png", dpi=150)
plt.show()

/var/folders/jb/prnhrlmn6ln8gp40kvxptyq80000gn/T/ipykernel_71453/728214780.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Donor type: impact and mix
Identifies **high-volume donor categories** for targeting and messaging.

In [6]:
hist["Donor Type"] = hist["Donor Type"].fillna("(missing)")
by_type = (
    hist.groupby("Donor Type")
    .agg(rescues=("Pounds Rescued", "count"), total_lbs=("Pounds Rescued", "sum"))
    .sort_values("total_lbs", ascending=False)
    .head(15)
)

fig, ax = plt.subplots()
ax.barh(by_type.index[::-1], (by_type["total_lbs"] / 1e6)[::-1], color="darkseagreen")
ax.set_xlabel("Million lbs (finished, historical)")
ax.set_title("Top donor types by total pounds rescued")
plt.tight_layout()
fig.savefig(OUT_DIR / "03_top_donor_types.png", dpi=150)
plt.show()
display(by_type.head(10))

/var/folders/jb/prnhrlmn6ln8gp40kvxptyq80000gn/T/ipykernel_71453/1083225969.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,rescues,total_lbs
Donor Type,,
Restaurant,72561,3417105.0
Other,4159,2101109.0
Office,14581,821981.0
(missing),4892,303624.0
School,744,36043.0


## 6. Partner performance snapshots
- **Top donors** by total pounds (relationship depth).
- **Top recipients** by volume (capacity / reliance).
- **Reliability:** coefficient of variation (CV) of monthly lbs per donor — high CV may deserve scheduling buffers or split pickups.

In [7]:
_hp = hist.copy()
_hp["Donor Name"] = _hp["Donor Name"].fillna("(unknown donor)")
_hp["Recipient Name"] = _hp["Recipient Name"].fillna("(unknown recipient)")

top_donors = (
    _hp.groupby("Donor Name")["Pounds Rescued"]
    .sum()
    .sort_values(ascending=False)
    .head(15)
)
top_recipients = (
    _hp.groupby("Recipient Name")["Pounds Rescued"]
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].barh(top_donors.index[::-1], top_donors.values[::-1] / 1e3, color="coral")
axes[0].set_title("Top donors (thousand lbs)")
axes[1].barh(top_recipients.index[::-1], top_recipients.values[::-1] / 1e3, color="slateblue")
axes[1].set_title("Top recipients (thousand lbs)")
plt.tight_layout()
fig.savefig(OUT_DIR / "04_top_donors_recipients.png", dpi=150)
plt.show()

h2 = _hp.copy()
h2["ym"] = h2["Rescue Date"].dt.to_period("M")
donor_month = h2.groupby(["Donor Name", "ym"])["Pounds Rescued"].sum().reset_index()
vol_stats = donor_month.groupby("Donor Name")["Pounds Rescued"].agg(["mean", "std", "count"])
vol_stats = vol_stats[vol_stats["count"] >= 6]
vol_stats["cv"] = vol_stats["std"] / vol_stats["mean"].replace(0, np.nan)
volatile = vol_stats.sort_values("cv", ascending=False).head(12)
display(volatile.round(1))

/var/folders/jb/prnhrlmn6ln8gp40kvxptyq80000gn/T/ipykernel_71453/1682243212.py:23: UserWarning: Tight layout not applied. tight_layout cannot make Axes width small enough to accommodate all Axes decorations
  plt.tight_layout()


/var/folders/jb/prnhrlmn6ln8gp40kvxptyq80000gn/T/ipykernel_71453/1682243212.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,mean,std,count,cv
Donor Name,,,,
Pret a Manger Franklin St.,1659.0,4862.8,32,2.9
Bridge Industrial,255.5,534.6,22,2.1
Shea & Company,116.6,208.1,13,1.8
Equator Design,718.3,1127.8,9,1.6
Tatte South End/Harrison Ave.,383.8,547.7,18,1.4
Gill Grilling at Sigma Chi MIT,98.0,135.6,8,1.4
"EZCater, Inc.",103.6,140.0,8,1.4
Rosa Mexicano Seaport,100.8,135.7,9,1.3
Levy Restaurants at Boston Convention and Exhibition Center,10727.9,13652.3,7,1.3


## 7. Zip codes: high-opportunity donor areas
Zip-level **total lbs** and **rescue count** highlight dense surplus pockets (optional: join census / income layers externally for a fuller “best fit” story).

In [8]:
def clean_zip(z):
    if pd.isna(z):
        return pd.NA
    s = str(z).split(".")[0].strip()
    if not s.isdigit():
        return pd.NA
    s = s.zfill(5)[-5:]
    return s


hz = hist.copy()
hz["donor_zip"] = hz["Donor Zipcode"].map(clean_zip)
zip_city = (
    hz.groupby(["Donor City", "donor_zip"])
    .agg(lbs=("Pounds Rescued", "sum"), n=("Pounds Rescued", "count"))
    .reset_index()
)
zip_city = zip_city.dropna(subset=["donor_zip"])

for city in ["Boston", "Chicago", "Los Angeles", "New York City"]:
    sub = zip_city[zip_city["Donor City"] == city].sort_values("lbs", ascending=False).head(8)
    print(f"\n{city} — top donor zips by total lbs")
    display(sub)


Boston — top donor zips by total lbs


,Donor City,donor_zip,lbs,n
6,Boston,02116,1462269.0,23561
19,Boston,02210,468816.0,10722
2,Boston,02110,428024.0,9232
20,Boston,02215,312895.0,7588
0,Boston,02108,295232.0,6101
21,Boston,02481,101248.0,1117
16,Boston,02139,59362.0,599
5,Boston,02115,59260.0,3178



Chicago — top donor zips by total lbs


,Donor City,donor_zip,lbs,n
37,Chicago,60638,1901526.0,1267
26,Chicago,60607,577535.0,11753
41,Chicago,60654,274941.0,5137
42,Chicago,60661,138028.0,2027
25,Chicago,60606,87699.0,1261
30,Chicago,60611,26522.0,705
27,Chicago,60608,21480.0,48
31,Chicago,60614,21176.0,838



Los Angeles — top donor zips by total lbs


,Donor City,donor_zip,lbs,n
43,Los Angeles,90004,95012.0,1852
45,Los Angeles,90024,38574.0,1120
54,Los Angeles,91604,19569.0,907
51,Los Angeles,90066,9235.0,385
49,Los Angeles,90038,5859.0,185
50,Los Angeles,90058,2280.0,1
48,Los Angeles,90034,1040.0,52
47,Los Angeles,90031,675.0,1



New York City — top donor zips by total lbs


,Donor City,donor_zip,lbs,n
57,New York City,10012,105201.0,3136
55,New York City,10009,27924.0,778
56,New York City,10011,26018.0,1000
58,New York City,10014,718.0,73


## 8. Simple pickup-volume forecast (next month)
Baseline **seasonal naive**: next month prediction = average lbs in that calendar month over history, per city. Good hackathon-visible baseline before fancier models.

*Capacity signal:* compare recent 3-month avg lbs to trailing 12-month avg per recipient (proxy for strain if recent >> baseline).

In [9]:
next_m = (as_of + pd.offsets.MonthBegin(1)).month
fc_rows = []
for city in ["Boston", "Chicago", "Los Angeles", "New York City"]:
    sub = hist[hist["Donor City"] == city]
    mavg = sub.groupby(sub["Rescue Date"].dt.month)["Pounds Rescued"].mean()
    pred = mavg.get(next_m, np.nan)
    scale = sub.groupby(sub["Rescue Date"].dt.to_period("M")).size().mean()
    fc_rows.append({"city": city, "forecast_month": next_m, "avg_lbs_per_rescue_that_month": pred, "avg_rescues_per_month_recent_proxy": scale})
fc = pd.DataFrame(fc_rows)
display(fc.round(1))

last3 = hist[hist["Rescue Date"] >= as_of - pd.DateOffset(months=3)]
prev12 = hist[hist["Rescue Date"] >= as_of - pd.DateOffset(months=15)]
prev12 = prev12[prev12["Rescue Date"] < as_of - pd.DateOffset(months=3)]
r3 = last3.groupby("Recipient Name")["Pounds Rescued"].sum()
r12 = prev12.groupby("Recipient Name")["Pounds Rescued"].sum() / 4
cap = pd.concat([r3.rename("last3mo"), r12.rename("prior_year_quarter_avg")], axis=1).dropna()
cap["ratio"] = cap["last3mo"] / cap["prior_year_quarter_avg"]
watch = cap.sort_values("ratio", ascending=False).head(15)
print("Recipients with last-3-mo volume >> historical quarter average (review capacity / scheduling)")
display(watch.round(1))

,city,forecast_month,avg_lbs_per_rescue_that_month,avg_rescues_per_month_recent_proxy
0,Boston,5,56.6,619.9
1,Chicago,5,127.8,500.9
2,Los Angeles,5,30.3,126.8
3,New York City,5,30.0,142.5


Recipients with last-3-mo volume >> historical quarter average (review capacity / scheduling)


,last3mo,prior_year_quarter_avg,ratio
Recipient Name,,,
Albany Park Community United Co-Op,2500.0,100.0,25.0
Catholic Charities at Yawkey Center,200.0,10.0,20.0
Casa Central,955.0,68.8,13.9
North Side Housing and Supportive Services,5720.0,660.0,8.7
Haley House,25360.0,5410.0,4.7
Farnsworth House Senior Center,1400.0,306.0,4.6
Lighthouse - Safe Place for Youth,80.0,21.2,3.8
Making The Right Turn,3576.0,954.0,3.7
The People Concern- Wilcox,840.0,300.0,2.8


## 9. Cost & routing optimization (batch peaks, same corridor)
**Goal:** reduce **trips and driver time** by scheduling **multi-stop runs** when several pickups fall in the **same area** and **same peak window**.

**Data limits:** there is **no lat/long**. We use **donor ZIP3** (first 3 digits) as a **corridor proxy**. `Starting_Time` in this extract is **coarse** (e.g. `00:00.0`, `30:00.0`); we still group on the **exact string** as a *declared pickup window* for “same-time” batching.

**Metrics:**
- **Weekday peaks** — where to add shift capacity.
- **Same-day + same ZIP3** — upper bound on mergeable pickups (one vehicle, multiple stops).
- **Same-day + ZIP3 + start slot** — stricter same-window batch candidates.
- **Volunteer-days with 2+ rescues** — replicate high-utilization routing patterns.


In [10]:
def clean_zip(z):
    if pd.isna(z):
        return pd.NA
    s = str(z).split(".")[0].strip()
    if not s.isdigit():
        return pd.NA
    return s.zfill(5)[-5:]


def donor_zip3(z):
    z = clean_zip(z)
    if pd.isna(z) or len(z) < 3:
        return pd.NA
    return z[:3]


opt = hist.copy()
opt["donor_zip"] = opt["Donor Zipcode"].map(clean_zip)
opt["zip3"] = opt["Donor Zipcode"].map(donor_zip3)
opt["dow"] = opt["Rescue Date"].dt.dayofweek
opt["start_slot"] = opt["Starting_Time"].astype(str).str.strip()
opt["Lead Rescuer Name"] = opt["Lead Rescuer Name"].fillna("").astype(str).str.strip()
opt.loc[opt["Lead Rescuer Name"].str.len() < 2, "Lead Rescuer Name"] = ""

# --- Peak demand by weekday (staffing / shift design)
dow_lab = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
peaks = (
    opt.groupby(["Donor City", "dow"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=range(7), fill_value=0)
)
peaks.columns = dow_lab

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(dow_lab))
width = 0.2
cities = ["Boston", "Chicago", "Los Angeles", "New York City"]
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
for i, city in enumerate(cities):
    if city not in peaks.index:
        continue
    ax.bar(x + (i - 1.5) * width, peaks.loc[city].values, width, label=city, color=colors[i])
ax.set_xticks(x)
ax.set_xticklabels(dow_lab)
ax.set_ylabel("Finished rescues (count)")
ax.set_title("Peak windows by weekday — align driver shifts with demand")
ax.legend(title="Donor city", fontsize=9)
plt.tight_layout()
fig.savefig(OUT_DIR / "05_dow_peaks_by_city.png", dpi=150)
plt.show()

# --- Batching: same calendar day + same ZIP3 (corridor proxy)
oz = opt.dropna(subset=["zip3"]).copy()
g_corridor = oz.groupby(["Rescue Date", "Donor City", "zip3"]).size().reset_index(name="n")
merge_groups = g_corridor[g_corridor["n"] > 1]
trip_legs_saved_upper = int((merge_groups["n"] - 1).sum())
sizes = oz.groupby(["Rescue Date", "Donor City", "zip3"])["Pounds Rescued"].transform("count")
oz["in_corridor_batch"] = sizes > 1
pct_in_batch = oz["in_corridor_batch"].mean()

# Stricter: same day + ZIP3 + reported start slot
gs = oz.groupby(["Rescue Date", "Donor City", "zip3", "start_slot"]).size().reset_index(name="n")
merge_slot = gs[gs["n"] > 1]
trip_legs_saved_slot = int((merge_slot["n"] - 1).sum())
slot_sizes = oz.groupby(["Rescue Date", "Donor City", "zip3", "start_slot"])["Pounds Rescued"].transform("count")
oz["in_slot_batch"] = slot_sizes > 1
pct_slot_batch = oz["in_slot_batch"].mean()

years = (opt["Rescue Date"].max() - opt["Rescue Date"].min()).days / 365.25
print(
    f"Historical window ~{years:.1f} years | finished rescues with ZIP3: {len(oz):,}"
)
print(
    f"Same-day same-ZIP3 multi-pickup groups: trip-start savings (upper bound): {trip_legs_saved_upper:,} "
    f"(~{trip_legs_saved_upper / max(years, 0.1):,.0f} / year)"
)
print(f"Share of rescues in any same-day ZIP3 batch (≥2 stops): {pct_in_batch:.1%}")
print(
    f"Same-day + ZIP3 + same start_slot: savings {trip_legs_saved_slot:,} "
    f"(~{trip_legs_saved_slot / max(years, 0.1):,.0f} / year) | share of rescues: {pct_slot_batch:.1%}"
)

by_city_batch = (
    oz.groupby("Donor City")["in_corridor_batch"].mean().reindex(cities).fillna(0)
)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(by_city_batch.index, by_city_batch.values * 100, color="teal", alpha=0.85)
ax.set_ylabel("% of rescues")
ax.set_title("Share of rescues in a same-day / same-ZIP3 batch (≥2 pickups)")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
fig.savefig(OUT_DIR / "06_batching_share_by_city.png", dpi=150)
plt.show()

# --- Volunteers already doing multiple rescues / day (scale their routes)
named = oz[oz["Lead Rescuer Name"] != ""]
vd = (
    named.groupby(["Lead Rescuer Name", "Rescue Date", "Donor City"])
    .agg(n_rescues=("Pounds Rescued", "count"), zip3s=("zip3", lambda s: s.nunique()))
    .reset_index()
)
multi = vd[vd["n_rescues"] >= 2]
print(
    f"Volunteer-days with 2+ finished rescues (named lead): {len(multi):,} "
    f"| avg rescues that day: {multi['n_rescues'].mean():.2f} "
    f"| avg distinct ZIP3: {multi['zip3s'].mean():.2f}"
)

# Top corridors (ZIP3) by mergeable extra stops — prioritize for route templates
corridor_rank = (
    merge_groups.groupby(["Donor City", "zip3"])
    .agg(batch_days=("Rescue Date", "count"), extra_stops=("n", lambda x: (x - 1).sum()))
    .sort_values("extra_stops", ascending=False)
)
print("Top corridor templates (city, ZIP3) by cumulative mergeable extra stops:")
display(corridor_rank.head(12).astype(int))


/var/folders/jb/prnhrlmn6ln8gp40kvxptyq80000gn/T/ipykernel_71453/1149949466.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/jb/prnhrlmn6ln8gp40kvxptyq80000gn/T/ipykernel_71453/1149949466.py:94: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Historical window ~8.5 years | finished rescues with ZIP3: 96,925
Same-day same-ZIP3 multi-pickup groups: trip-start savings (upper bound): 89,068 (~10,477 / year)
Share of rescues in any same-day ZIP3 batch (≥2 stops): 98.9%
Same-day + ZIP3 + same start_slot: savings 85,175 (~10,019 / year) | share of rescues: 96.9%


Volunteer-days with 2+ finished rescues (named lead): 9,018 | avg rescues that day: 9.99 | avg distinct ZIP3: 1.15
Top corridor templates (city, ZIP3) by cumulative mergeable extra stops:


batch_days  extra_stops
Donor City    zip3                         
Boston        021         2826        41494
Chicago       606         1107        22333
Boston        022         1642        16256
New York City 100          662         4138
Los Angeles   900          318         3117
Boston        024          140          925
Los Angeles   916          113          794
              914            1           11

## 10. Talking points for your deck (edit with your narrative)
1. **Trends:** `01_monthly_pounds_by_city.png` — growth and city mix.
2. **Seasonality:** `02_seasonality_index.png` — pre-stage volunteers in high-index months.
3. **Targeting:** `03_top_donor_types.png` + zip tables — recruit similar donors.
4. **Partnership health:** `04_top_donors_recipients.png` + CV table — deepen accounts; stabilize volatile donors.
5. **Cost / routing:** `05_dow_peaks_by_city.png` + `06_batching_share_by_city.png` — **shift design** + **ZIP3 batching** to cut duplicate trip starts; quote **trip savings upper bound** from §9 printout.
6. **Operations:** completion rate + recipient capacity watchlist — fewer failed rescues.
7. **Stretch:** geocode addresses or merge census layers on `donor_zip` for finer “same direction” than ZIP3.

**Figures:** `figures/` for slides.
